In [ ]:
PARAPHRASE_PROMPT = """You are given a spoken description of the Cookie Theft picture.

Write one alternative version that sounds like a real person describing the same scene in a similar way.

Guidelines:
- Keep the same general content and level of detail.
- Use different wording where natural.
- Keep it conversational and speech-like.
- Minor changes in order and phrasing are fine.
- Do not add important new details that were not in the original.
- Do not make it much more polished, coherent, or complete than the original.
- Keep the length reasonably similar.

Return only the rewritten transcript.

Transcript:
{text}"""

print("Paraphrase prompt defined.")

In [ ]:
import anthropic
import pandas as pd
import time
from tqdm import tqdm

client = anthropic.Anthropic()
real_df = pd.read_csv("alz-speech.csv")

records = []

for _, row in tqdm(real_df.iterrows(), total=len(real_df), desc="Paraphrasing"):
    prompt = PARAPHRASE_PROMPT.format(text=row["transcript"])
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=512,
            messages=[{"role": "user", "content": prompt}],
        )
        transcript = response.content[0].text.strip().replace("\n", " ")
    except Exception as e:
        print(f"Error at subject={row['subject']}: {e}")
        transcript = ""

    records.append({
        "subject":         row["subject"],
        "sex":             row["sex"],
        "age":             int(row["age"]),
        "ad":              int(row["ad"]),
        "transcript":      transcript,
        "paraphrase_type": "P",
    })
    time.sleep(0.02)

para_df = pd.DataFrame(records)
para_df.to_csv("alz-speech-paraphrased.csv", index=False)
print(f"Saved {len(para_df)} records to alz-speech-paraphrased.csv")
print(f"AD balance: {para_df['ad'].value_counts().to_dict()}")
para_df.head()